## E3

## Mision 1: modelacion de entidades

Tip: ver bien el archivo, entender qué está pasando, y así darse una idea de qué se puede buscar.

In [50]:
class Actor:
    def __init__(self, nombre):
        self.nombre = nombre
        self.peliculas = []

    def agregar_pelicula(self, pelicula):
        self.peliculas.append(pelicula)
    
class Genero:
    def __init__(self, nombre):
        self.nombre = nombre
        self.peliculas = []

    def agregar_pelicula(self, pelicula):
        self.peliculas.append(pelicula)

class Pelicula:
    def __init__(self, titulo, anio):
        self.titulo = titulo
        self.anio = anio
        self.generos = []
        self.actores = []

    def agregar_actor(self, actor):
        self.actores.append(actor)
        actor.agregar_pelicula(self)

    def agregar_genero(self, genero):
        self.generos.append(genero)
        genero.agregar_pelicula(self)

## Mision 2: carga de datos

In [51]:
import json

with open('movies.json', encoding='utf8') as f:
    data = json.load(f)

In [52]:
print(f"Se han cargado {len(data)} películas del archivo JSON.")

Se han cargado 28795 películas del archivo JSON.


In [53]:
# imprimir 30 películas para verificar que se cargaron correctamente
for i in range(30):
    print(data[i])

{'title': 'After Dark in Central Park', 'year': 1900, 'cast': [], 'genres': []}
{'title': "Boarding School Girls' Pajama Parade", 'year': 1900, 'cast': [], 'genres': []}
{'title': "Buffalo Bill's Wild West Parad", 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Caught', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Clowns Spinning Hats', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Capture of Boer Battery by British', 'year': 1900, 'cast': [], 'genres': ['Short', 'Documentary']}
{'title': 'The Enchanted Drawing', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Feeding Sea Lions', 'year': 1900, 'cast': ['Paul Boyton'], 'genres': []}
{'title': 'How to Make a Fat Wife Out of Two Lean Ones', 'year': 1900, 'cast': [], 'genres': ['Comedy']}
{'title': 'New Life Rescue', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'New Morning Bath', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Searching Ruins on Broadway, Galveston, for Dead Bodies', 'year': 1900, 'cast': [], 'gen

In [54]:
actores_dict = {}
generos_dict = {}
peliculas = []

In [55]:
# funcion auxiliar
def es_actor_valido(nombre):
    if not nombre:
        return False
    
    nombre = nombre.strip()

    if nombre == "":
        return False
    
    basura = {"and", ".", ",", "(voice)"}
    if nombre.lower() in basura:
        return False
    
    # descartar anotaciones entre paréntesis
    if nombre.startswith("(") and nombre.endswith(")"):
        return False
    
    # opcional: evitar nombres demasiado cortos, si es que hay casos de eso
    if len(nombre) < 3:
        return False
    
    return True


In [56]:
peliculas_vistas = set()

for movie in data:
    cast = []
    generos = []

    # actores
    for a in movie["cast"]:
        if a not in actores_dict:
            actores_dict[a] = Actor(a)

        actor = actores_dict[a]
        cast.append(actor)

    # géneros
    for g in movie["genres"]:
        if g not in generos_dict:
            generos_dict[g] = Genero(g)

        genero = generos_dict[g]
        generos.append(genero)

    clave = (movie["title"], movie["year"], tuple(cast))

    if clave not in peliculas_vistas:
        pelicula = Pelicula(movie["title"], movie["year"])
        
        for genero in generos:
            pelicula.agregar_genero(genero)

        for actor in cast:
            pelicula.agregar_actor(actor)

        peliculas.append(pelicula)
        peliculas_vistas.add(clave)
    else:
        print(f"Película duplicada encontrada: {movie['title']} ({movie['year']})")

Película duplicada encontrada: The Prince and Betty (1919)
Película duplicada encontrada: The Virtuous Thief (1919)
Película duplicada encontrada: The Wise Kid (1922)
Película duplicada encontrada: The White Sister (1923)
Película duplicada encontrada: Wild Bill Hickok Rides (1942)
Película duplicada encontrada: The Rough, Tough West (1952)


In [57]:
print(f"dict_peliculas tiene {len(peliculas)} películas únicas.")

dict_peliculas tiene 28789 películas únicas.


Vemos que 28789 + 6 = 28795. Así que está bien

## Mision 3: Consultas sobre los datos

1) Encuentre los 5 generos mas populares.

In [58]:
conteo = [(g.nombre, len(g.peliculas)) for g in generos_dict.values()]
top_generos = sorted(conteo, key=lambda x: x[1], reverse=True)[:5]

# tip: revisar funciones lambda y sorted para agilar el conteo y ordenamiento
for genero, count in top_generos:
    print(f"Género: {genero}, cantidad de películas: {count}")

Género: Drama, cantidad de películas: 8742
Género: Comedy, cantidad de películas: 7361
Género: Western, cantidad de películas: 3011
Género: Crime, cantidad de películas: 1499
Género: Horror, cantidad de películas: 1166


2)  Encuentre los 3 años con mas peliculas estrenadas

In [59]:
conteo_anios = {} 

for p in peliculas:
    if p.anio not in conteo_anios:
        conteo_anios[p.anio] = 0
    conteo_anios[p.anio] += 1

# ordenar
top_anios = sorted(conteo_anios.items(), key=lambda x: x[1], reverse=True)[:3]

for anio, count in top_anios:
    print(f"Año: {anio}, cantidad de películas: {count}")

Año: 1919, cantidad de películas: 632
Año: 1925, cantidad de películas: 572
Año: 1936, cantidad de películas: 504


3) Encuentre a los 5 actores con la trayectoria mas larga, es decir, mayor cantidad de años actuando.

In [60]:
trayectoria = []

for actor in actores_dict.values():
    if not es_actor_valido(actor.nombre):
        # print(f"Actor inválido descartado: '{actor.nombre}'")
        continue
    years = [p.anio for p in actor.peliculas]
    if years:
        trayectoria.append((actor.nombre, max(years) - min(years)))

top_actores = sorted(trayectoria, key=lambda x: x[1], reverse=True)[:5]
for actor, duracion in top_actores:
    print(f"Actor: {actor}, duración de trayectoria: {duracion} años")

Actor: Harrison Ford, duración de trayectoria: 98 años
Actor: Gloria Stuart, duración de trayectoria: 80 años
Actor: Lillian Gish, duración de trayectoria: 75 años
Actor: Kenny Baker, duración de trayectoria: 75 años
Actor: Mickey Rooney, duración de trayectoria: 74 años


Prueben descomentando la linea del actor inválido, y se darán cuenta de por qué decidí crear la función actor válido.

4)  Encuentre el reparto de una pelicula (2 o mas actores) que se haya repetido completo en otras la mayor
cantidad de veces

In [61]:
conteo_repartos = {}

for p in peliculas:
    reparto_completo = []

    for a in p.actores:
        nombre = a.nombre.strip()
        if es_actor_valido(nombre):
            reparto_completo.append(nombre)

    reparto_completo = tuple(reparto_completo)

    if len(reparto_completo) >= 2:
        if reparto_completo not in conteo_repartos:
            conteo_repartos[reparto_completo] = 0
        conteo_repartos[reparto_completo] += 1

max_reparto = None
max_count = 0

for reparto, count in conteo_repartos.items():
    if count > max_count:
        max_count = count
        max_reparto = reparto

print("Reparto completo más repetido:", max_reparto)
print("Cantidad de veces:", max_count)

Reparto completo más repetido: ('Harold Lloyd', 'Bebe Daniels')
Cantidad de veces: 44
